<a href="https://colab.research.google.com/github/MissSamyuktha/Sales-Forecasting-End-to-End-ML/blob/main/Sales_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SuperKart Sales Prediction Model Deployment

# Deployed Application Links on HuggingFace

Backend Flask App: [Backend Flask App](https://huggingface.co/spaces/MissSamyuktha/sales-forecast-backend)

Streamlit App: [Frontend Streamlit UI](https://huggingface.co/spaces/MissSamyuktha/sales-forecast-frontend)


# Business Context
A sales forecast predicts future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action.

Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecast process has many benefits, which include improved decision-making about the future and the reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establishes benchmarks that can be used to assess trends in the future.

# Objective
SuperKart is a retail chain operating supermarkets and food marts across various tier cities, offering a wide range of products. To optimize its inventory management and make informed decisions around regional sales strategies, SuperKart wants to accurately forecast the sales revenue of its outlets for the upcoming quarter.

To operationalize these insights at scale, the company has partnered with a data science firm, not just to build a predictive model based on historical sales data but also to develop and deploy a robust forecasting solution that can be integrated into SuperKart’s decision-making systems and used across its network of stores.

# Data Dictionary
The data contains the different attributes of the various products and stores.

- Product_Id: Unique identifier of each product, each identifier having two letters at the beginning, followed by a number
- Product_Weight: Weight of each product
Product_Sugar_Content: Sugar content of each product, like low sugar, regular, and no sugar
- Product_Allocated_Area: Ratio of the allocated display area of each product to the total display area of all the products in a store
- Product_Type: Broad category for each product like meat, snack foods, hard drinks, dairy, canned, soft drinks, health and hygiene, baking goods, bread, breakfast, frozen foods, fruits and vegetables, household, seafood, starchy foods, others
- Product_MRP: Maximum retail price of each product
- Store_Id: Unique identifier of each store
- Store_Establishment_Year: Year in which the store was established
- Store_Size: Size of the store, depending on sq. feet, like high, medium, and low
- Store_Location_City_Type: Type of city in which the store is located, like Tier 1, Tier 2, and Tier 3. Tier 1 consists of cities where the standard of living is comparatively higher than that of its Tier 2 and Tier 3 counterparts
- Store_Type: Type of store depending on the products that are being sold there, like Departmental Store, Supermarket Type 1, Supermarket Type 2, and Food Mart
- Product_Store_Sales_Total: Total revenue generated by the sale of that particular product in that particular store

## Installing and Importing Necessary Libraries

In [ ]:
#Installing the libraries with the specified versions
!pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 joblib==1.4.2 flask==2.2.2 xgboost==2.1.4 requests==2.32.4 huggingface_hub==0.30.1 -q

In [ ]:
import pandas as pd   # for data manipulation
import numpy as np    # for numpy array scientific calculation
import matplotlib.pyplot as plt # plotting data visually
import seaborn as sns  # plotting data with better interactive graphical interface

# data preprocessing and building model pipelines
from sklearn.model_selection import train_test_split    # splitting data into train and test sets
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # for z score feature scaling and categorical data encoding
from sklearn.pipeline import Pipeline, make_pipeline    # for model pipelines
from sklearn.compose import make_column_transformer     # for applying transformations to columns

# model building
from sklearn.linear_model import LinearRegression       # linear regression model
from sklearn.metrics import mean_absolute_error as mae, mean_squared_error as mse, root_mean_squared_error as rmse, r2_score #accuracy metrics

from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree  #regression using decision trees cart model
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, AdaBoostRegressor as abr, GradientBoostingRegressor as gbr #ensemble models

from xgboost import XGBRegressor # extreme gradient boost

# hyperparameter tuning
from sklearn.model_selection import GridSearchCV

# model serialization
import joblib

# for creating a folder
import os

# for huggingface authentication and to upload files to spaces
from huggingface_hub import login, HfApi

# to ignore warnings
import warnings
warnings.filterwarnings('ignore')


# Data Overview

In [ ]:
# loading our dataset into pandas dataframe
data = pd.read_csv("/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project/SuperKart.csv")

In [ ]:
# shape of our data
data.shape
print(f"The dataset has {data.shape[0]} rows.")
print(f"The dataset has {data.shape[1]} columns.")

In [ ]:
# Quick glance of data
data.head()

In [ ]:
# data information
data.info()

The data is loaded perfectly.  
It has 8763 rows and 12 columns with no null entries.  
There are 7 categorical and 5 numerical columns.  

In [ ]:
# let's see if there are any duplicate entries
data.duplicated().sum()

There are no duplicate records found.

In [ ]:
# let's cross check again for any null values
data.isnull().sum()

There are no missing values found.

# Data Preprocessing

In [ ]:
# our tarhet variable for this model
target = 'Product_Store_Sales_Total'

## Feature Engineering

In [ ]:
# Let's explore Store_Establishment_Year looks
data['Store_Establishment_Year'].nunique()

There are only 4 year values repeating in this column.

In [ ]:
# let's see what those 4 unique year entries repeating
data['Store_Establishment_Year'].unique()

This 'Store_Establishment_Year' can very be categorized as categorical data.

In [ ]:
# let's convert int year values to object type
data['Store_Establishment_Year'] = data['Store_Establishment_Year'].astype('object')

In [ ]:
numerical_features = data.select_dtypes(exclude=[object]).drop(columns=['Product_Store_Sales_Total']).columns.to_list()
print("numerical_features:", numerical_features)

categorical_features = data.select_dtypes(include=[object]).columns.to_list()
print("categorical_features:", categorical_features)

# Data Overview (contd.)

## Statistical Summary

In [ ]:
# summary stats of numerical features
data[numerical_features].describe().T

Product_weight data is normally distributed no significat skewness, ranging from 4 to 22 with 75% of data falling below 14.18.  

Product_Allocated_Area is right skewed, ranging from 0.004 to 0.298, but 75% of data falls below 0.096, indicating there are few very large values, draging its raigh tail.

Product_MRP is normally distributed with no significant skewness, ranging from 31 to 266, withn75% dat falling below 167.585.

In [ ]:
# summary stats of categorical features
data[categorical_features].describe().T

- Product_Id is unique to each row indicating 8796 different product items in the data.
- Product_Sugar_Content has 4 types with 'Low Sugar' as highest sold.
- Product_Type has 16 types with 'Fruits and Vegetables' type highly sold.
- Store_Id has 4 types of stores with OUT004 as most sold store.
- Store_Establishment_Year: There are only 4 years where stores were established with 2009 as year with most establishments.
- Store_Size: 3 sizes, Medium stores sold most.
- Store_Location_City_Type: 3 types, Tier 2 sold highest.
Store_Type: 4 types, Supermarket Type 2 made most sales.


In [ ]:
# summary stats of target variable
data[target].describe()

Target is more or less normally distributed, values ranging from 33 to 8000. 75% data falls below 4145, indicating there are few large purchases.

# Data Preprocessing (contd..)

## Checking for data accuracy

In [ ]:
data[data['Product_Weight'] < 0]

In [ ]:
data[data['Product_Allocated_Area'] < 0]

In [ ]:
data[data['Product_MRP'] < 0]

There are no negative values in the numerical features. The data is accurate.

# Univariate Analysis

In [ ]:
# let's see how Product_Weight data is distributed
plt.title("Distribution of Product_Weight")
sns.histplot(data=data, x='Product_Weight', kde=True);

In [ ]:
# boxplot distribution of Product_Weight
plt.title("Boxplot Distribution of Product_Weight")
sns.boxplot(data=data, x='Product_Weight');

Product_Weight data is more or less normally distributed.

Some of the data falls outside the whiskers. These outliers are not necessarily anamalies. These outliers could represent:
- Lightweight items (e.g., spices, small snacks)
- Heavier items (e.g., bulk goods, household products)

In [ ]:
# let's see how Product_Allocated_Area is distributed
plt.title("Distribution of Product_Allocated_Area")
sns.histplot(data=data, x='Product_Allocated_Area', kde=True);

In [ ]:
# boxplot distribution of Product_Allocated_Area
plt.title("Boxplot Distribution of Product_Allocated_Area")
sns.boxplot(data=data, x='Product_Allocated_Area');

Product_Allocated_Area data is rightly skewed, with few large values dragging it right tail, may need log transformations for model training.

Boxplot confirms right skewed values, shows numerous outliers beyond 0.20. These could represent promotional or high-visibility products.


In [ ]:
# let's see how Product_MRP is distributed
plt.title("Distribution of Product_MRP")
sns.histplot(data=data, x='Product_MRP', kde=True);

In [ ]:
# boxplot distribution of Product_MRP
plt.title("Boxplot Distribution of Product_MRP")
sns.boxplot(data=data, x='Product_MRP');

Product_MRP is normally distributed, centered around MRP of 150, suggesting that most products are priced near the middle range, with fewer items at very low or very high prices.

The boxplot confirms this: the box is symmetric, and whiskers are evenly spread, with a presence of a few outliers exist below 50 and above 250.  
These could represent:
- Budget items or Sale of old stock (e.g., small packs, basic essentials, or promotional goods).
- Premium products (e.g., bulk items, imported goods, high-value items).

In [ ]:
# let's see how Store_Establishment_Year looks
plt.title("How many stores established in each year")
sns.countplot(data=data, x='Store_Establishment_Year');

More than 50% stores are newest, established in year 2009.

# Data Preprocessing (contd..)

In [ ]:
# let's see how Product_Sugar_Content looks
data['Product_Sugar_Content'].unique()

The data is inconsistent here, both reg, and Regular represent same type.

In [ ]:
# replacing reg with Regular foe data consistency
data['Product_Sugar_Content']= data['Product_Sugar_Content'].replace('reg', 'Regular')

# Univariate Analysis (contd..)

In [ ]:
# let's see how different types in Product_Sugar_Content are distributed
plt.title("Sales according to sugar content in the products")
sns.countplot(data=data, x='Product_Sugar_Content');

More than 50% are solely from low sugar products, indicating peoples' health choices.

In [ ]:
# lets dig deeper into Product_Type data
data['Product_Type'].unique()

In [ ]:
# let's see sales of different product types
plt.figure(figsize=(12, 6))
plt.title("Sales of Different Product Types")
sns.countplot(data=data, x='Product_Type')
plt.xticks(rotation =60, ha='right')
plt.tight_layout()
plt.show()

Most sold product type is fruits and vegetables, second to it is snack foods.

In [ ]:
# digging deeper into stores
data['Store_Id'].unique()

In [ ]:
# sales from different stores
plt.title("sales from Different Stores")
sns.countplot(data=data, x='Store_Id');

Highest sales were from OUT004 store_id

In [ ]:
# let's explore Store_Size feature
data['Store_Size'].unique()

In [ ]:
# Sales across different store sizes
plt.title("Sales across different store sizes")
sns.countplot(data=data, x='Store_Size');

Medium size stores made most sales.

In [ ]:
# let's explore Store_Location_City_Type feature
data['Store_Location_City_Type'].unique()

In [ ]:
# sales across different city types
plt.title("Sales across city types")
sns.countplot(data=data, x='Store_Location_City_Type');

Around 75% sales occured from Tier 2 city stores.

In [ ]:
# exploring Store_Type feature
data['Store_Type'].unique()

In [ ]:
# sales across different store types
plt.figure(figsize=(8, 6))
plt.title("Sales across store types")
sns.countplot(data=data, x='Store_Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Maximum sales are from Supermarket Type 2 stores

In [ ]:
# let's see how our target variable is distributed
plt.title("Distribution of Product Sales")
sns.histplot(data=data, x='Product_Store_Sales_Total', kde=True);

In [ ]:
# boxplot distribution of target variable
plt.title("Boxplot Distribution of Product Sales")
sns.boxplot(data=data, x='Product_Store_Sales_Total');

Our target variable ie Product_Store_Sales_Total is normally distributed, centered around 3500–4000 whicg is ideal for regression models as they perform well when the target is symmetrically distributed.

This suggests that most store-product combinations generate moderate sales, with fewer extreme cases.

The boxplot shows a few outliers below ~700 and above ~6300. These could represent:
- Niche products with low demand.
- Or High-performing products in flagship stores or during promotions.

# Bivariate Analysis

In [ ]:
# correlation heatmap of numerical variables with target
plt.title("Correlation Heatmap of Numerical Features & Target")
sns.heatmap(data.select_dtypes(exclude=[object]).corr(), annot=True);

Product_Weight and Product_MRP are moderately correlated(0.53).  
Product_Weight and Target are highly correlated(0.74).    
Product_MRP and Target are highly correlated(0.79).    
There are no other sinificant correlations found.  

In [ ]:
# pairplot of numerical features
print("Pairplot of Numerical Features & Target")
sns.pairplot(data.select_dtypes(exclude=[object]));

Following corretaions are founf from the pairplot:
- Product_Weight and Product_MRP are correlated.
- Product_Weight and Target are correlated.
- Product_MRP and Target are correlated.

There are no other significant correlations found.


In [ ]:
# Product_Sugar_Content vs target
plt.title("Product_Sugar_Content vs Target Sales")
sns.boxplot(data=data, x='Product_Sugar_Content', y='Product_Store_Sales_Total');

The boxplot shows that median sales per product are roughly equal across "Low Sugar", "Regular", and "No Sugar".

So while Low Sugar dominates in count, it doesn't necessarily outperform in per-product sales.

Outliers could represent niche products with extreme performance or promotional spikes or underperforming SKUs.

**What This Means for Modeling?**  
Sugar content alone may not be a strong predictor of sales, but it could be useful when combined with other features.

In [ ]:
# Product_Type vs target
plt.figure(figsize=(12, 6))
plt.title("Product_Type vs Target Sales")
sns.boxplot(data=data, x='Product_Type', y='Product_Store_Sales_Total')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

Similar Median Sales Across Most Categories falling within a narrow band around 3500–4000, suggesting that no single product type dominates in per-unit sales, at least in terms of central tendency.

Wide Spread in Some Categories like Household, Snack Foods, and Fruits and Vegetables show larger interquartile ranges, indicating greater variability in sales. These may include both high-volume and niche products.

Presence of outliers could represent seasonal spikes, premium or bulk items, promotional campaigns.

**Implications for Modeling**  
Product_Type could be a valuable categorical feature as it captures sales variability across categories. May interact meaningfully with other features.

In [ ]:
# Store_Id vs target
plt.title("Store_Id vs Target Sales")
sns.boxplot(data=data, x='Store_Id', y='Product_Store_Sales_Total');

OUT003 stands out with the Highest Median Sales and a Wide interquartile range. This store likely serves a high-demand region or has better product placement and inventory turnover.

OUT002 Has the Lowest Sales and Least Variability, indicates consistently low sales possibly due to smaller size, lower footfall, or limited product range.

OUT001 is mid performer with decent median sales with few high sales and many low value sales. Wide range, could have mix of different prodcuts in sales.

OUT004 is a mid performer with presence of outliers. Also  other stores show a few outliers, could be promotional spikes, bulk purchases, or high-value items.

**Implication for Modeling**  
Store_Id is a highly informative categorical feature that captures sales performance and variability.

In [ ]:
# Store_Establishment_Year vs target
plt.title("Store_Establishment_Year vs Target Sales")
sns.boxplot(data=data, x='Store_Establishment_Year', y='Product_Store_Sales_Total');

Stores Established in 1987 and 1999 Perform Better showing higher median sales and wider interquartile ranges, indicating that older stores may have stronger customer bases, better inventory practices, or more optimized operations.

1998 has the lowest median and several low outliers.

2009 stores show lower variability, suggesting more consistent but modest performance possibly newer stores still scaling up.

**Implication for Modeling**  
Store_Establishment_Year is a meaningful categorical feature that captures store maturity and its impact on sales.

In [ ]:
# Store_Size vs target
plt.title("Store_Size vs Target Sales")
sns.boxplot(data=data, x='Store_Size', y='Product_Store_Sales_Total');

High-Sized Stores Have the Highest Median Sales indicating that larger stores may benefit from better inventory, more footfall, or broader product variety.

Medium Stores Show the Widest Spread and many outliers suggesting they serve diverse customer bases or have inconsistent performance across locations.

Small Stores Lag Behind, have the lowest median and tightest spread indicating limited sales potential, possibly due to space constraints or fewer SKUs.

**Implication for Modeling**  
Store_Size is a meaningful categorical feature that captures operational scale and its impact on sales.

In [ ]:
# Store_Location_City_Type vs target
plt.title("Store_Location_City_Type vs Target Sales")
sns.boxplot(data=data, x='Store_Location_City_Type', y='Product_Store_Sales_Total');

Tier 1 Cities Lead in Median Sales, have the highest median and a wide spread indicating stronger purchasing power, higher footfall, or better product mix in metropolitan areas.

Tier 2 Cities Are Competitive, show slightly lower median than Tier 1 but similar variability. Suggests Tier 2 cities are still strong contributors to overall revenue possibly due to volume.

Tier 3 Cities Lag Behind, have the lowest median and tightest spread. Indicates limited sales potential possibly due to smaller markets or fewer high-value products.

**Implication for Modeling**  
Store_Location_City_Type is a valuable categorical feature that captures regional economic tiering and its impact on sales.

In [ ]:
# Store_Type vs target
plt.figure(figsize=(8,6))
plt.title("Store_Type vs Target Sales")
sns.boxplot(data=data, x='Store_Type', y='Product_Store_Sales_Total')
plt.xticks(rotation=42, ha='right')
plt.tight_layout()
plt.show()

Departmental Stores Lead in Median Sales, has the highest median and a wide spread. Suggests strong performance, possibly due to diverse product offerings or higher customer loyalty.

Supermarket Type1 Performs Well, shows relatively high median sales with a tighter interquartile range. Indicates consistent performance across locations.

Supermarket Type 2 has Moderate Median Sales, lower median than Departmental Stores and Supermarket Type 1, suggest inconsistent performance across locations or product mixes but still contributes meaningfully.

Food Mart Lags Behind, has the lowest median and compact distribution likely smaller format with limited inventory and lower footfall.

**Implication for Modeling**  
Store_Type is a valuable categorical feature that captures operational format and its impact on sales.

# Data Preprocessing (contd..)

## Outlier Detection

All the Product_Weight, Product_Allocated_Area, Product_MRP, Product_Store_Sales_Total columns have some extreme case values but those can be justified.

- Product_weight extremes are most likely lightweight items (e.g., spices, small snacks) or heavier items (e.g., bulk goods, household products).
- Product_Allocated_Area has some right skewed extreme values, could represent promotional or high-visibility products.
- Product_MRP's extremes are most likely budget items or Sale of old stock (e.g., small packs, basic essentials, or promotional goods) or premium products (e.g., bulk items, imported goods, high-value items).
- Target(Product_Store_Sales_Total) shows a few extremes, that are most likely niche products with low demand or High-performing products in flagship stores or during promotions.

Therefore the data is most likely valid and **no need to treat any outliers here**.

## Feature Engineering

### log transformation

In [ ]:
# making a copy of data
df = data.copy()

In [ ]:
df['log_Product_Allocated_Area'] = np.log(df['Product_Allocated_Area'])

In [ ]:
# distribution of log_Product_Allocated_Area
plt.title("Distribution of log_Product_Allocated_Area")
sns.histplot(data=df, x='log_Product_Allocated_Area', kde=True);

log_Product_Allocated_Area is less skewed than Product_Allocated_Area, now to be used in the model training.

In [ ]:
numerical_features

In [ ]:
# replacing Product_Allocated_Area with log_Product_Allocated_Area
numerical_features[1] = 'log_Product_Allocated_Area'
numerical_features

In [ ]:
categorical_features

In [ ]:
# dropping Product_Id as it does not contribute to the model
categorical_features= ['Product_Sugar_Content', 'Product_Type', 'Store_Id', 'Store_Establishment_Year', 'Store_Size', 'Store_Location_City_Type', 'Store_Type']

### Data preprocessing for model builing

In [ ]:
# Define predictor matrix (X) using selected numerical and categorical features
X = df[numerical_features + categorical_features]
y = df['Product_Store_Sales_Total']

In [ ]:
X.columns

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
# splitting data into train and test data(20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) #randomstate for reproducibility

In [ ]:
print(f"shape of X_train: {X_train.shape}")
print(f"shape of y_train: {y_train.shape}")
print(f"shape of X_test: {X_test.shape}")
print(f"shape of y_test: {y_test.shape}")

In [ ]:
# Create a preprocessing pipeline for numerical and categorical features
preprocessor = make_column_transformer(
    (StandardScaler(), numerical_features),  # Scale numeric features to z-score normalization
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)  # Encode categorical features as one-hot vectors
)

# Model Building

# Model Evaluation Metrics

**1. Root Mean Square Error (RMSE)**
- Measures average prediction error in the same units as the target (e.g., sales ₹).
- Penalizes large errors more heavily due to squaring, useful when big mistakes are costly.
- Easy to interpret: a lower RMSE means predictions are closer to actual values.

**2. R² Score (Coefficient of Determination)**
- Explains the proportion of variance in the target variable that the model captures.
- Ranges from 0 to 1 (or negative if worse than baseline):  
  - 1.0 = perfect prediction
  - 0.0 = model explains none of the variance
- Helps assess model fit and compare across models.

**Why They Work Well Together?**  
RMSE gives you a scale-sensitive error metric.

R² gives you a scale-independent goodness-of-fit.

Using both provides a balanced view: accuracy + explanatory power.

## Linear Regression Model

In [ ]:
# initializing linear regression
lreg = LinearRegression()

In [ ]:
# Create a machine learning pipeline with preprocessing and model training steps
lreg_pipeline = make_pipeline(preprocessor, lreg)

In [ ]:
lreg_pipeline.fit(X_train, y_train)

In [ ]:
lreg_model = lreg_pipeline.named_steps['linearregression']
print(lreg_model.coef_)

In [ ]:
print(lreg_model.intercept_)

In [ ]:
y_pred_train_lreg = lreg_pipeline.predict(X_train)
y_pred_test_lreg = lreg_pipeline.predict(X_test)

In [ ]:
lreg_train_error = rmse(y_train, y_pred_train_lreg)
print(lreg_train_error)

lreg_test_error = rmse(y_test, y_pred_test_lreg)
print(lreg_test_error)

In [ ]:
lreg_train_r2 = r2_score(y_train, y_pred_train_lreg)
print(lreg_train_r2)

lreg_test_r2 = r2_score(y_test, y_pred_test_lreg)
print(lreg_test_r2)

## Linear Regression Model Tuning

In [ ]:
from sklearn.linear_model import Lasso
lasso = Lasso()
lasso_pipeline = make_pipeline(preprocessor, lasso)

In [ ]:
params = {'lasso__alpha':[0.1, 1, 10, 100]}
lasso_grid = GridSearchCV(estimator=lasso_pipeline, param_grid=params, scoring='r2', n_jobs=-1, cv=5, verbose=2)
lasso_grid.fit(X_train, y_train)

In [ ]:
lasso_model = lasso_grid.best_estimator_.named_steps['lasso']
print(lasso_model.coef_)

In [ ]:
print("Number of non-zero coefficients:", np.sum(lasso_model.coef_!= 0))

In [ ]:
print(lasso_model.intercept_)

In [ ]:
# Get feature names after preprocessing
feature_names = lasso_grid.best_estimator_.named_steps['columntransformer'].get_feature_names_out()

# Pair features with coefficients
coefs = lasso_model.coef_
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefs
})

# Sort by absolute importance
coef_df = coef_df.reindex(coef_df.Coefficient.abs().sort_values(ascending=False).index)
coef_df

In [ ]:
top_features = coef_df.head(5)

plt.figure(figsize=(10,6))
plt.barh(top_features["Feature"], top_features["Coefficient"], color="skyblue")
plt.xlabel("Coefficient Value")
plt.title("Top 5 Lasso Feature Importances")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
lasso_grid.best_params_

In [ ]:
lasso_grid.best_score_

In [ ]:
y_pred_train_lasso = lasso_grid.predict(X_train)
y_pred_test_lasso = lasso_grid.predict(X_test)

In [ ]:
lasso_train_error = rmse(y_train, y_pred_train_lasso)
print("lasso train error:", lasso_train_error)

lasso_test_error = rmse(y_test, y_pred_test_lasso)
print("lasso test error:", lasso_test_error)

In [ ]:
lasso_train_r2 = r2_score(y_train, y_pred_train_lasso)
print("lasso train r2:", lasso_train_r2)

lasso_test_r2 = r2_score(y_test, y_pred_test_lasso)
print("lasso test r2:", lasso_test_r2)

## Ridge Regression Model Tuning

In [ ]:
# initialise ridge regression
from sklearn.linear_model import Ridge
ridge = Ridge()

In [ ]:
ridge_pipeline = make_pipeline(preprocessor, ridge)

In [ ]:
ridge.get_params()

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
params = {'ridge__alpha':[0.1, 1, 10, 100]}
random_search = RandomizedSearchCV(estimator=ridge_pipeline, param_distributions=params, scoring='r2', n_jobs=-1, verbose=2)
random_search.fit(X_train, y_train)

In [ ]:
print("Best parameters:", random_search.best_params_)
print("Best CV R²:", random_search.best_score_)

In [ ]:
ridge_model = random_search.best_estimator_.named_steps['ridge']
print("Model Coefficients:", ridge_model.coef_)
print("Model Intercept:", ridge_model.intercept_)

In [ ]:
print("Number of non-zero coefficients:", np.sum(ridge_model.coef_!= 0))

In [ ]:
# Get feature names after preprocessing
feature_names = random_search.best_estimator_.named_steps['columntransformer'].get_feature_names_out()

# Pair features with coefficients
coefs = ridge_model.coef_
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefs
})

# Sort by absolute importance
coef_df = coef_df.reindex(coef_df.Coefficient.abs().sort_values(ascending=False).index)
coef_df.head(20)  # top 20 features

In [ ]:
top_features = coef_df.head(5)

plt.figure(figsize=(10,6))
plt.barh(top_features["Feature"], top_features["Coefficient"], color="skyblue")
plt.xlabel("Coefficient Value")
plt.title("Top 5 Ridge Feature Importances")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
y_pred_train_ridge = random_search.predict(X_train)
y_pred_test_ridge = random_search.predict(X_test)

In [ ]:
ridge_train_error = rmse(y_train, y_pred_train_ridge)
print("Ridge train error:", ridge_train_error)

ridge_test_error = rmse(y_test, y_pred_test_ridge)
print("Ridge test error:", ridge_test_error)

In [ ]:
ridge_train_r2 = r2_score(y_train, y_pred_train_ridge)
print("Ridge train r2:", ridge_train_r2)

ridge_test_r2 = r2_score(y_test, y_pred_test_ridge)
print("Ridge test r2:", ridge_test_r2)

## ElasticNet Model tuning

In [ ]:
from sklearn.linear_model import ElasticNet
# initialize the model
enet = ElasticNet(random_state=42)

In [ ]:
enet_pipeline = make_pipeline(preprocessor, enet)

In [ ]:
enet.get_params()

In [ ]:
params = {
    'elasticnet__alpha':[0.01, 0.1, 1, 10, 100],
    'elasticnet__l1_ratio':np.arange(0.3, 0.8, 0.1)
}

enet_rsearch = RandomizedSearchCV(estimator=enet_pipeline, param_distributions=params, scoring='r2', n_jobs=-1, verbose=2, cv=5)
enet_rsearch.fit(X_train, y_train)

In [ ]:
print("Best parameters:", enet_rsearch.best_params_)
print("Best CV R²:", enet_rsearch.best_score_)

In [ ]:
enet_model = enet_rsearch.best_estimator_.named_steps['elasticnet']
print("Model Coefficients:", enet_model.coef_)
print("Model Intercept:", enet_model.intercept_)

In [ ]:
print("Number of non-zero coefficients:", np.sum(enet_model.coef_!= 0))

In [ ]:
# Get feature names after preprocessing
feature_names = enet_rsearch.best_estimator_.named_steps['columntransformer'].get_feature_names_out()

# Pair features with coefficients
coefs = enet_model.coef_
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefs
})

# Sort by absolute importance
coef_df = coef_df.reindex(coef_df.Coefficient.abs().sort_values(ascending=False).index)
coef_df.head(20)  # top 20 features


In [ ]:
top_features = coef_df.head(5)

plt.figure(figsize=(10,6))
plt.barh(top_features["Feature"], top_features["Coefficient"], color="skyblue")
plt.xlabel("Coefficient Value")
plt.title("Top 5 ElasticNet Feature Importances")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
y_pred_train_enet = enet_rsearch.predict(X_train)
y_pred_test_enet = enet_rsearch.predict(X_test)

In [ ]:
enet_train_error = rmse(y_train, y_pred_train_enet)
print("ElasticNet train error:", enet_train_error)

enet_test_error = rmse(y_test, y_pred_test_enet)
print("ElasticNet test error:", enet_test_error)

In [ ]:
enet_train_r2 = r2_score(y_train, y_pred_train_enet)
print("ElasticNet train r2:", enet_train_r2)

enet_test_r2 = r2_score(y_test, y_pred_test_enet)
print("ElasticNet test r2:", enet_test_r2)

## SGD Regressor Model

In [ ]:
from sklearn.linear_model import SGDRegressor
sgd = SGDRegressor(random_state=42)

In [ ]:
sgd_pipeline = make_pipeline(preprocessor, sgd)

In [ ]:
sgd.get_params()

In [ ]:
params = {
    'sgdregressor__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100],
    'sgdregressor__l1_ratio': np.arange(0.1, 0.8, 0.1),
    'sgdregressor__penalty': ['l1', 'l2', 'elasticnet'],
    'sgdregressor__learning_rate': ['constant', 'optimal', 'invscaling', 'adaptive'],
    'sgdregressor__early_stopping': [True],
    'sgdregressor__validation_fraction': [0.2]
}

sgd_rsearch = RandomizedSearchCV(
    estimator=sgd_pipeline,
    param_distributions=params,
    n_jobs=-1,
    verbose=2,
    cv=5,
    random_state=42
)

sgd_rsearch.fit(X_train, y_train)


In [ ]:
print("Best parameters:", sgd_rsearch.best_params_)
print("Best CV R²:", sgd_rsearch.best_score_)

In [ ]:
sgd_model = sgd_rsearch.best_estimator_.named_steps['sgdregressor']
print("Model Coefficients:", sgd_model.coef_)
print("Model Intercept:", sgd_model.intercept_)

In [ ]:
print("Number of non-zero coefficients:", np.sum(sgd_model.coef_!= 0))

In [ ]:
# Get feature names after preprocessing
feature_names = sgd_rsearch.best_estimator_.named_steps['columntransformer'].get_feature_names_out()

# Pair features with coefficients
coefs = sgd_model.coef_
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefs
})

# Sort by absolute importance
coef_df = coef_df.reindex(coef_df.Coefficient.abs().sort_values(ascending=False).index)
coef_df.head(20)  # top 20 features

In [ ]:
top_features = coef_df.head(5)

plt.figure(figsize=(10,6))
plt.barh(top_features["Feature"], top_features["Coefficient"], color="skyblue")
plt.xlabel("Coefficient Value")
plt.title("Top 5 SGD Feature Importances")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
y_pred_train_sgd = sgd_rsearch.predict(X_train)
y_pred_test_sgd = sgd_rsearch.predict(X_test)

In [ ]:
sgd_train_error = rmse(y_train, y_pred_train_sgd)
print("SGD train error:", sgd_train_error)

sgd_test_error = rmse(y_test, y_pred_test_sgd)
print("SGD test error:", sgd_test_error)

In [ ]:
sgd_train_r2 = r2_score(y_train, y_pred_train_sgd)
print("SGD train r2:", sgd_train_r2)

sgd_test_r2 = r2_score(y_test, y_pred_test_sgd)
print("SGD test r2:", sgd_test_r2)

## SVM Regression

In [ ]:
from sklearn.svm import SVR
svr = SVR()

In [ ]:
SVR().get_params()

In [ ]:
svr_pipeline = make_pipeline(preprocessor, svr)

In [ ]:
params = {
    'svr__kernel':['linear', 'rbf', 'poly', 'sigmoid'],
    'svr__gamma':['scale', 'auto'],
    'svr__C':[0.01, 0.1, 1, 10, 100],
    'svr__degree':[2,3,4]
}

from sklearn.model_selection import RandomizedSearchCV
svr_rsearch = RandomizedSearchCV(estimator=svr_pipeline,param_distributions=params, scoring='r2', cv=5, n_jobs=-1, verbose=2, random_state=42)
svr_rsearch.fit(X_train, y_train)

In [ ]:
svr_rsearch.best_params_

In [ ]:
svr_rsearch.best_score_

In [ ]:
svr_model = svr_rsearch.best_estimator_.named_steps['svr']

In [ ]:
y_pred_train_svr = svr_rsearch.predict(X_train)
y_pred_test_svr = svr_rsearch.predict(X_test)

In [ ]:
svr_train_error = rmse(y_train, y_pred_train_svr)
print("SVR train error:", svr_train_error)

svr_test_error = rmse(y_test, y_pred_test_svr)
print("SVR test error:", svr_test_error)

In [ ]:
svr_train_r2 = r2_score(y_train, y_pred_train_svr)
print("SVR train r2:", svr_train_r2)

svr_test_r2 = r2_score(y_test, y_pred_test_svr)
print("SVR test r2:", svr_test_r2)

In [ ]:
import shap

# Transform the training and test data using the preprocessor
X_train_transformed = svr_rsearch.best_estimator_.named_steps['columntransformer'].transform(X_train)
X_test_transformed = svr_rsearch.best_estimator_.named_steps['columntransformer'].transform(X_test)

# Use KernelExplainer for non-linear SVR
explainer = shap.KernelExplainer(svr_model.predict, X_train_transformed[:100])  # use a subset for background
shap_values = explainer.shap_values(X_test_transformed[:50])  # explain a subset for speed

shap.summary_plot(shap_values, X_test_transformed[:50])


## KNN Model

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
knn = KNeighborsRegressor()

In [ ]:
knn_pipeline = make_pipeline(preprocessor, knn)

In [ ]:
knn.get_params()

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

params = {'kneighborsregressor__n_neighbors':[5,6,7,8],
          'kneighborsregressor__algorithm':['auto', 'kd_tree', 'ball_tree'],
          'kneighborsregressor__weights':['uniform', 'distance'],
          'kneighborsregressor__leaf_size':[30,40,50]
          }
knn_rsearch = RandomizedSearchCV(estimator=knn_pipeline, param_distributions=params, scoring='r2', n_jobs=-1, verbose=2, cv=5)
knn_rsearch.fit(X_train, y_train)

In [ ]:
knn_rsearch.best_params_

In [ ]:
knn_rsearch.best_score_

In [ ]:
knn_model = knn_rsearch.best_estimator_.named_steps['kneighborsregressor']

In [ ]:
y_pred_train_knn = knn_rsearch.predict(X_train)
y_pred_test_knn = knn_rsearch.predict(X_test)

In [ ]:
knn_train_error = rmse(y_train, y_pred_train_knn)
print("KNN train error:", knn_train_error)

knn_test_error = rmse(y_test, y_pred_test_knn)
print("KNN test error:", knn_test_error)

In [ ]:
knn_train_r2 = r2_score(y_train, y_pred_train_knn)
print("KNN train r2:", knn_train_r2)

knn_test_r2 = r2_score(y_test, y_pred_test_knn)
print("KNN test r2:", knn_test_r2)

In [ ]:
import shap

# Transform the training and test data using the preprocessor
X_train_transformed = knn_rsearch.best_estimator_.named_steps['columntransformer'].transform(X_train)
X_test_transformed = knn_rsearch.best_estimator_.named_steps['columntransformer'].transform(X_test)

# Use KernelExplainer for non-linear SVR
explainer = shap.KernelExplainer(knn_model.predict, X_train_transformed[:100])  # use a subset for background
shap_values = explainer.shap_values(X_test_transformed[:50])  # explain a subset for speed

shap.summary_plot(shap_values, X_test_transformed[:50])

In [ ]:
# storing the model pipeline
knn_model_pipeline = knn_rsearch.best_estimator_

### Model Serialization

In [ ]:
project_path = '/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project'
os.chdir(project_path)

In [ ]:
joblib.dump(knn_model_pipeline, 'sales_forecasting_knn.joblib')

## Model Deployment

In [ ]:
import os
project_path = '/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project'
os.chdir(project_path)

In [ ]:
# creating a folder for backend deployment files
os.makedirs('backend', exist_ok=True)

In [ ]:
os.chdir('/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project/backend')

In [ ]:
%%writefile app.py
import os
import json
import joblib
import numpy as np
import pandas as pd
from flask import Flask, request, jsonify

# Flask app
app = Flask(__name__)

# Load model pipeline (ColumnTransformer + KNN)
# load the trained sales forecasting model
model = joblib.load('sales_forecasting_knn.joblib')

# Expected columns as in training
NUMERIC_FEATURES = ['Product_Weight', 'log_Product_Allocated_Area', 'Product_MRP']
CATEGORICAL_FEATURES = ['Product_Sugar_Content', 'Product_Type', 'Store_Id',
                        'Store_Establishment_Year', 'Store_Size',
                        'Store_Location_City_Type', 'Store_Type']
INPUT_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

@app.route("/", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": "sales_forecasting_knn"})

@app.route("/predict", methods=["POST"])
def predict():
    try:
        payload = request.get_json(force=True)
        # Accept single record or batch
        if isinstance(payload, dict):
            data = pd.DataFrame([payload], columns=INPUT_COLUMNS)
        elif isinstance(payload, list):
            data = pd.DataFrame(payload, columns=INPUT_COLUMNS)
        else:
            return jsonify({"error": "Invalid payload type"}), 400

        # Ensure log_Product_Allocated_Area exists (if user passed raw Product_Allocated_Area optionally)
        if "Product_Allocated_Area" in data.columns and "log_Product_Allocated_Area" not in data.columns:
            data["log_Product_Allocated_Area"] = np.log(data["Product_Allocated_Area"])
            data.drop(columns=["Product_Allocated_Area"], inplace=True)

        # Reorder columns
        data = data[INPUT_COLUMNS]
        preds = model.predict(data)
        return jsonify({"predictions": preds.tolist()})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=int(os.environ.get("PORT", 7860)))


In [ ]:
%%writefile requirements.txt
pandas==2.2.2
numpy==2.0.2
scikit-learn==1.6.1
joblib==1.4.2
Werkzeug==2.2.2
flask==2.2.2
gunicorn==20.1.0
requests==2.32.4
uvicorn[standard]


In [ ]:
%%writefile Dockerfile
# HF Spaces Docker uses Python slim images well
FROM python:3.10-slim

WORKDIR /app

# System deps (optional, helpful for numpy/pandas)
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

# Copy files
COPY requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# Copy your Flask app and model
COPY app.py /app/app.py
COPY sales_forecasting_knn.joblib /app/sales_forecasting_knn.joblib

# Default port on Spaces is 7860
ENV PORT=7860
EXPOSE 7860

CMD ["python", "app.py"]


In [ ]:
# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

from google.colab import userdata
access_key = userdata.get('hf_token')

repo_id = "MissSamyuktha/forecast-backend"  # Hugging Face space id

# Login to Hugging Face platform with the access token
login(token=access_key)

# initialize the apiA
api = HfApi()

# Upload Flask app files stored in the folder called backend_files
api.upload_folder(folder_path='/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project/backend', # Local folder path
                  repo_id=repo_id,    ## Hugging face space id
                  repo_type="space"  # Hugging face repo type "space"
)



In [ ]:
os.chdir('/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project')

In [ ]:
# creating a folder for backend deployment files
os.makedirs('frontend', exist_ok=True)

In [ ]:
os.chdir('/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project/frontend')

In [ ]:
!pip install --upgrade streamlit==1.51.0


In [ ]:
%%writefile app.py
import os
import requests
import streamlit as st
import numpy as np
import pandas as pd

st.set_page_config(page_title="SuperKart Sales Forecasting", page_icon="🛒", layout="centered")
st.title("🛒 SuperKart Sales Forecasting")

# Backend URL (set to your HF backend Space URL)
BACKEND_URL = 'https://MissSamyuktha-forecast-backend.hf.space'
#BACKEND_URL = st.secrets.get("BACKEND_URL", os.environ.get("BACKEND_URL", ""))

st.caption("Enter product and store details to forecast sales.")

with st.form("input_form"):
    # Numeric inputs
    product_weight = st.number_input("Product Weight", min_value=0.0, step=0.1)
    product_alloc_area = st.number_input("Product Allocated Area (original, will be logged)", min_value=0.0001, step=0.001)
    product_mrp = st.number_input("Product MRP", min_value=0.0, step=1.0)

    # Categorical inputs
    sugar_content = st.selectbox("Product Sugar Content", ["Low Sugar", "Regular", "No Sugar"])
    product_type = st.selectbox("Product Type", [
        "Frozen Foods","Dairy","Canned","Baking Goods","Health and Hygiene","Snack Foods","Meat",
        "Household","Hard Drinks","Fruits and Vegetables","Breads","Soft Drinks","Breakfast","Others",
        "Starchy Foods","Seafood"
    ])
    store_id = st.selectbox("Store Id", ["OUT001","OUT002","OUT003","OUT004"])
    store_est_year = st.selectbox("Store Establishment Year", [1987, 1998, 1999, 2009])
    store_size = st.selectbox("Store Size", ["Small","Medium","High"])
    city_type = st.selectbox("Store Location City Type", ["Tier 1","Tier 2","Tier 3"])
    store_type = st.selectbox("Store Type", ["Departmental Store","Supermarket Type1","Supermarket Type2","Food Mart"])

    submitted = st.form_submit_button("Forecast")

if submitted:
    if not BACKEND_URL:
        st.error("Backend URL not set. Please configure BACKEND_URL in Streamlit secrets or environment.")
    else:
        payload = {
            "Product_Weight": product_weight,
            "log_Product_Allocated_Area": float(np.log(product_alloc_area)),
            "Product_MRP": product_mrp,
            "Product_Sugar_Content": sugar_content,
            "Product_Type": product_type,
            "Store_Id": store_id,
            "Store_Establishment_Year": store_est_year,
            "Store_Size": store_size,
            "Store_Location_City_Type": city_type,
            "Store_Type": store_type
        }
        try:
            resp = requests.post(f"{BACKEND_URL}/predict", json=payload, timeout=30)
            if resp.status_code == 200:
                pred = resp.json()["predictions"][0]
                st.success(f"Forecasted Sales: {pred:,.2f}")
            else:
                st.error(f"Backend error: {resp.text}")
        except Exception as e:
            st.error(f"Request failed: {e}")

st.divider()
st.caption("Built with a KNN pipeline (StandardScaler + OneHotEncoder + KNN Regressor).")


In [ ]:
os.getcwd()

In [ ]:
%%writefile requirements.txt
streamlit==1.51.0
requests==2.32.4
numpy==2.0.2
pandas==2.2.2


In [ ]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

# Copy requirements first (better for caching)
COPY requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# Copy the app code
COPY app.py /app/app.py

# Spaces streamlit runner expects `python app.py` or `streamlit run app.py`
ENV PORT=7860
EXPOSE 7860

# Use streamlit runner
CMD ["streamlit", "run", "app.py", "--server.port=7860", "--server.address=0.0.0.0"]


In [ ]:
# for hugging face space authentication to upload files
from huggingface_hub import HfApi

repo_id = "MissSamyuktha/forecast-frontend"  #  Hugging Face space id

# Initialize the API
api = HfApi()

# Upload Streamlit app files stored in the folder called deployment_files
api.upload_folder(
    folder_path="/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project/frontend",  # Local folder path
    repo_id=repo_id,  # Hugging face space id
    repo_type="space",  # Hugging face repo type "space"
)

## Decision Tree Regressor Model

### Model Pipeline

In [ ]:
# initializing decision tree regressor
dtr = DecisionTreeRegressor(random_state=42)

In [ ]:
# Create a machine learning pipeline with preprocessing and model training steps
dtr_pipeline = make_pipeline(
    preprocessor,  # Preprocesses numerical and categorical features
    dtr            # DecisionTreeRegressor for model training
)

### Model Training

In [ ]:
# Train the model pipeline on the training data
dtr_pipeline.fit(X_train, y_train)

In [ ]:
# make predictions on train data
y_pred_train = dtr_pipeline.predict(X_train)

# make predictions on test data
y_pred_test = dtr_pipeline.predict(X_test)

### Model Performance Evaluation

In [ ]:
# Root mean square error for train data predictions
dtr_train_error = rmse(y_train, y_pred_train)
print(f"Root mean square error for train data predictions: {dtr_train_error}")

In [ ]:
# Root mean square error for test data predictions
dtr_test_error = rmse(y_test, y_pred_test)
print(f"Root mean square error for test data predictions: {dtr_test_error}")

In [ ]:
# r2_score: how well our model captures the variability in train data
dtr_train_r2 = r2_score(y_train, y_pred_train)
print(f"The dtr model explains {dtr_train_r2*100}% train data variance")

In [ ]:
# r2_score: how well our model captures the variability in test data
dtr_test_r2 = r2_score(y_test, y_pred_test)
print(f"The dtr model explains {dtr_test_r2*100}% test data variance")

### Feature Importance

In [ ]:
# Extract trained model from pipeline
dtr_model = dtr_pipeline.named_steps['decisiontreeregressor']

# Get feature names after preprocessing
# This works only if you're using OneHotEncoder and StandardScaler in a ColumnTransformer
encoded_feature_names = dtr_pipeline.named_steps['columntransformer'].get_feature_names_out()

# Get importances
importances = dtr_model.feature_importances_

# Create a DataFrame for sorting and plotting
feat_imp_df = pd.DataFrame({
    'Feature': encoded_feature_names,
    'Importance': importances
})

# Sort and select top 5
top5 = feat_imp_df.sort_values(by='Importance', ascending=False).head(5)

# Plot
plt.figure(figsize=(8, 5))
plt.barh(top5['Feature'], top5['Importance'], color='skyblue')
plt.xlabel("Feature Importance")
plt.title("Top 5 Important Features (Decision Tree)")
plt.gca().invert_yaxis()  # Highest importance at the top
plt.tight_layout()
plt.show()

Top 5 features contributing to our decision tree model: Product_MRP, Product_Weight, Store_Size_Small, Store_Id_OUT002, Store_Type_Food_Mart.

This simple decision tree regressor model(dtr_pipeline) clearly overfits the train data.

There is absolute zero error and explains 100% variance of train data.

Explains ~92% of test data which is good but the model needs to be pruned through hyperparameter tuning.


## RandomForestRegressor

In [ ]:
# initializing random forest regressor
rfreg = RandomForestRegressor(random_state=42)

In [ ]:
# creating model pipeline
rfreg_pipeline = make_pipeline(
                  preprocessor,
                  rfreg
)

In [ ]:
# training the model pipeline on train data
rfreg_pipeline.fit(X_train, y_train)

In [ ]:
# make predictions on train data
y_pred_train_rf = rfreg_pipeline.predict(X_train)

# make predictions on test data
y_pred_test_rf = rfreg_pipeline.predict(X_test)

### Feature Importance

In [ ]:
# Extract trained model from pipeline
rf_model = rfreg_pipeline.named_steps['randomforestregressor']

# Get feature names after preprocessing
# This works only if you're using OneHotEncoder and StandardScaler in a ColumnTransformer
encoded_feature_names = rfreg_pipeline.named_steps['columntransformer'].get_feature_names_out()

# Get importances
importances = rf_model.feature_importances_

# Create a DataFrame for sorting and plotting
feat_imp_df = pd.DataFrame({
    'Feature': encoded_feature_names,
    'Importance': importances
})

# Sort and select top 5
top5 = feat_imp_df.sort_values(by='Importance', ascending=False).head(5)

# Plot
plt.figure(figsize=(8, 5))
plt.barh(top5['Feature'], top5['Importance'], color='skyblue')
plt.xlabel("Feature Importance")
plt.title("Top 5 Important Features (Random Forest)")
plt.gca().invert_yaxis()  # Highest importance at the top
plt.tight_layout()
plt.show()


### Model Performance Evaluation

In [ ]:
# Root mean square error for train data predictions
rfreg_train_error = rmse(y_train, y_pred_train_rf)
print(f"Root mean square error for train data predictions: {rfreg_train_error}")

In [ ]:
# Root mean square error for test data predictions
rfreg_test_error = rmse(y_test, y_pred_test_rf)
print(f"Root mean square error for test data predictions: {rfreg_test_error}")

In [ ]:
# r2_score: how well our model captures the variability in train data
rfreg_train_r2 = r2_score(y_train, y_pred_train_rf)
print(f"The rfreg model explains {rfreg_train_r2*100}% train data variance")

In [ ]:
# r2_score: how well our model captures the variability in test data
rfreg_test_r2 = r2_score(y_test, y_pred_test_rf)
print(f"The rfreg model explains {rfreg_test_r2*100}% test data variance")

Top 5 features contributing to our random forest model: Product_MRP, Product_Weight, Store_Size_Small, Store_Id_OUT002, Store_Type_Food_Mart.

The model explains 99% on train data and 93% on test data which is good but some overfitting is present but better than decision tree dtr model earlier.

The error on test also decreased from earlier.

could apply hyperparameter tuning for reducing overfit and better score.

## XGBoostRegressor Model

In [ ]:
# initializing xgboostregressor
xgb = XGBRegressor(tree_method='hist', device='cuda', random_state=42)

In [ ]:
# creating model pipeline
xgb_pipeline = make_pipeline(
                preprocessor,
                xgb
)

In [ ]:
print(xgb_pipeline.named_steps.keys())


In [ ]:
# training model pipeline on train data
xgb_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_train_xgb = xgb_pipeline.predict(X_train) #model predictions for train data
y_pred_test_xgb = xgb_pipeline.predict(X_test) # model predictions for test data


### Feature Importance

In [ ]:
# Extract trained model from pipeline
xgb_model = xgb_pipeline.named_steps['xgbregressor']

# Get feature names after preprocessing
# This works only if you're using OneHotEncoder and StandardScaler in a ColumnTransformer
encoded_feature_names = xgb_pipeline.named_steps['columntransformer'].get_feature_names_out()

# Get importances
importances = xgb_model.feature_importances_

# Create a DataFrame for sorting and plotting
feat_imp_df = pd.DataFrame({
    'Feature': encoded_feature_names,
    'Importance': importances
})

# Sort and select top 5
top5 = feat_imp_df.sort_values(by='Importance', ascending=False).head(5)

# Plot
plt.figure(figsize=(8, 5))
plt.barh(top5['Feature'], top5['Importance'], color='skyblue')
plt.xlabel("Feature Importance")
plt.title("Top 5 Important Features (XGBoost)")
plt.gca().invert_yaxis()  # Highest importance at the top
plt.tight_layout()
plt.show()


### Model Performance Evaluation

In [ ]:
# Root mean square error for train data predictions
xgb_train_error = rmse(y_train, y_pred_train_xgb)
print(f"Root mean square error for train data predictions: {xgb_train_error}")

In [ ]:
# Root mean square error for test data predictions
xgb_test_error = rmse(y_test, y_pred_test_xgb)
print(f"Root mean square error for test data predictions: {xgb_test_error}")

In [ ]:
# r2_score: how well our model captures the variability in train data
xgb_train_r2 = r2_score(y_train, y_pred_train_xgb)
print(f"The xgb model explains {xgb_train_r2*100}% train data variance")

In [ ]:
# r2_score: how well our model captures the variability in test data
xgb_test_r2 = r2_score(y_test, y_pred_test_xgb)
print(f"The xgb model explains {xgb_test_r2*100}% test data variance")

All 4 store ids and product_mrp contributed the most to our XGBoost model.

xgb model explains 97.4% of train data and 92.2% of test data, means reduced overfitting and better generalization of the model.

Same can be explained from error rates which are train:170.8, test:296.5, difference reduced by large margin from earlier models.

Performance can be improved further by hyperparameter tuning.

# Hyperparameter Tuning

## Decision Tree Regressor Tuned Model

In [ ]:
# define parameter grid
param_grid = {
    'decisiontreeregressor__max_depth': [5, 10, 20, None],
    'decisiontreeregressor__min_samples_split': [2, 5, 10],
    'decisiontreeregressor__min_samples_leaf': [1, 2, 4],
    'decisiontreeregressor__criterion': ['squared_error', 'absolute_error']
}


In [ ]:
# setting up gridsearchcv
grid_search_dtr = GridSearchCV(
    estimator=dtr_pipeline,
    param_grid=param_grid,
    scoring='r2',
    cv=3,
    n_jobs=-1,
    verbose=2
)


In [ ]:
# run the gridsearchcv
grid_search_dtr.fit(X_train, y_train)

In [ ]:
print("Best Parameters:", grid_search_dtr.best_params_)
print("Best Score:", grid_search_dtr.best_score_)

In [ ]:
# best model
dtr_tuned = grid_search_dtr.best_estimator_

In [ ]:
# model predictions on train data
y_pred_train_dtr_tuned = dtr_tuned.predict(X_train)

# model predictions on test data
y_pred_test_dtr_tuned = dtr_tuned.predict(X_test)

### Model Performance Evaluation

In [ ]:
# Root mean square error for train data predictions
dtr_tuned_train_error = rmse(y_train, y_pred_train_dtr_tuned)
print(f"Root mean square error for train data predictions: {dtr_tuned_train_error}")

In [ ]:
# Root mean square error for test data predictions
dtr_tuned_test_error = rmse(y_test, y_pred_test_dtr_tuned)
print(f"Root mean square error for test data predictions: {dtr_tuned_test_error}")

In [ ]:
# r2_score: how well our model captures the variability in train data
dtr_tuned_train_r2 = r2_score(y_train, y_pred_train_dtr_tuned)
print(f"The dtr model explains {dtr_tuned_train_r2*100}% train data variance")

In [ ]:
# r2_score: how well our model captures the variability in test data
dtr_tuned_test_r2 = r2_score(y_test, y_pred_test_dtr_tuned)
print(f"The dtr model explains {dtr_tuned_test_r2*100}% test data variance")

The dtr tuned model explains over 99% train data but 92% test data, means still overfits.

Also the error is large for test data predictions.

we shall consider xgboost tuned model now.

## XGBoost Tuned Model

In [ ]:
# Define Parameter Grid
param_grid = {
    'xgbregressor__n_estimators': [100, 200],
    'xgbregressor__max_depth': [3, 6, 10],
    'xgbregressor__learning_rate': [0.05, 0.1],
    'xgbregressor__subsample': [0.8, 1.0],
    'xgbregressor__colsample_bytree': [0.8, 1.0]
}


In [ ]:
# Set Up GridSearchCV
grid_search_xgb = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    scoring='r2',      # or 'neg_root_mean_squared_error'
    cv=3,
    n_jobs=-1,
    verbose=2
)


In [ ]:
# Run the Grid Search
grid_search_xgb.fit(X_train, y_train)

In [ ]:
# View Best Parameters and Score
print("Best Parameters:", grid_search_xgb.best_params_)
print("Best Score:", grid_search_xgb.best_score_)

In [ ]:
# best model
xgb_tuned = grid_search_xgb.best_estimator_

In [ ]:
# model predictions for train data
y_pred_train_xgb_tuned = xgb_tuned.predict(X_train)

# model predictions for test data
y_pred_test_xgb_tuned = xgb_tuned.predict(X_test)

### Model Performance Evaluation

In [ ]:
# Root mean square error for train data predictions
xgb_tuned_train_error = rmse(y_train, y_pred_train_xgb_tuned)
print(f"Root mean square error for train data predictions: {xgb_tuned_train_error}")

In [ ]:
# Root mean square error for test data predictions
xgb_tuned_test_error = rmse(y_test, y_pred_test_xgb_tuned)
print(f"Root mean square error for test data predictions: {xgb_tuned_test_error}")

In [ ]:
# r2_score: how well our model captures the variability in train data
xgb_tuned_train_r2 = r2_score(y_train, y_pred_train_xgb_tuned)
print(f"The xgb model explains {xgb_tuned_train_r2*100}% train data variance")

In [ ]:
# r2_score: how well our model captures the variability in test data
xgb_tuned_test_r2 = r2_score(y_test, y_pred_test_xgb_tuned)
print(f"The xgb model explains {xgb_tuned_test_r2*100}% test data variance")

Lower RMSE than all earlier models on both train and test means xgb_tuned makes smaller prediction errors.

Higher R² means it explains more variance in both train(98.8%) and test(92.9%) datasets.

Smaller gap between train and test R² suggests better generalization and less overfitting.

# Model Performance comparison

In [ ]:
metrics_df = pd.DataFrame(
            {
                'Model': ['dtr', 'rforest', 'xgb', 'dtr_tuned', 'xgb_tuned'],
                'rmse_train':[0.0, 102.92, 170.79, 90.61, 116.90],
                'rmse_test': [304.48, 280.42, 296.49, 300.07, 284.75],
                'rmse_diff': [304.48, 177.5 , 125.7 , 209.46, 167.85],
                'r2_train': [100.0, 99.06, 97.42, 99.27, 98.79],
                'r2_test': [91.87, 93.10, 92.29, 92.10, 92.89],
                'r2_diff': [8.13, 5.96, 5.13, 7.17, 5.9 ]
                }
)

In [ ]:
metrics_df

**xgb_tuned** is the best model because:
- Lower RMSE than all earlier models on both train and test means xgb_tuned makes smaller prediction errors.
- Higher R² means it explains more variance in both train(98.8%) and test(92.9%) datasets.
- Smaller gap between train and test r2_score suggests better generalization and less overfitting.

Although simple xgb model has least gap between train and test rmse, and r2_score, because of the above mentioned reasons xgb_tuned is picked out as the best model.

# Model Serialization

In [ ]:
project_path = '/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project'
os.chdir(project_path)

In [ ]:
joblib.dump(xgb_tuned, 'sales_forecast_xgb_tuned.joblib')

### Loading joblib model to make predictions

In [ ]:
project_path = '/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project'
os.chdir(project_path)

In [ ]:
model = joblib.load('sales_forecast_xgb_tuned.joblib') # loading the model

In [ ]:
# model predictions on test data
predictions = model.predict(X_test)

### Model Evaluation

In [ ]:
rmse_error = rmse(y_test, predictions)
print("Root Mean Square Error on test data:", rmse_error)

In [ ]:
r2 = r2_score(y_test, predictions)
print(f"The best model explains {r2*100}% variance of test data")

The chosen best model xgb_tuned explains 92.9% variability of test data with low error rate of 284.75.

# Model Deployment

## Backend Deployment

In [ ]:
os.chdir('/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project')

In [ ]:
# creating a folder for backend deployment files
os.makedirs('backend_files', exist_ok=True)

In [ ]:
os.chdir('/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project/backend_files')

In [ ]:
joblib.dump(model, 'sales_forecast_xgb_tuned.joblib') # storing joblib model in our backend_files folder

### We Set up a Hugging Face Docker Space for the Backend [HuggingFace Spaces](https://huggingface.co/spaces/)

In [ ]:
pip install --force-reinstall Flask==2.2.2 Werkzeug==2.2.2

In [ ]:
import flask, werkzeug
print(flask.__version__, werkzeug.__version__)
print(flask.__file__)
print(werkzeug.__file__)


### Flask Web Framework

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
import numpy as np
from flask import Flask, request, jsonify

# initialize the flask app with a name
app = Flask("SuperKart Sales Forecasting Application")

# load the trained sales forecasting model
model = joblib.load('sales_forecast_xgb_tuned.joblib')

# define a route for the home page
@app.get('/')
def home():
  return "Welcome to SuperKart Sales Forecasting Application"

# define an endpoint to predict sales forecast for a single Product_Store_Sales_Total
@app.post('/v1/sales')
def sales_forecast():
  # get json data from the request
  sales_data = request.get_json()

  # extract relevant sales info from the input data
  sample = {
            'Product_Weight': sales_data['Product_Weight'],
            'log_Product_Allocated_Area': np.log(sales_data['Product_Allocated_Area']),
            'Product_MRP': sales_data['Product_MRP'],
            'Product_Sugar_Content': sales_data['Product_Sugar_Content'],
            'Product_Type': sales_data['Product_Type'],
            'Store_Id': sales_data['Store_Id'],
            'Store_Establishment_Year': str(sales_data['Store_Establishment_Year']),
            'Store_Size': sales_data['Store_Size'],
            'Store_Location_City_Type': sales_data['Store_Location_City_Type'],
            'Store_Type': sales_data['Store_Type']
  }

  # convert the extracted data into a DataFrame
  input_data = pd.DataFrame([sample])

  # Make sales forecasting using the trained model
  prediction = model.predict(input_data)

  # Return the prediction as a JSON response
  return jsonify({'Forecasted Sales': float(prediction[0])})


# Define an endpoint to forecast sales for a batch of sales data
@app.post('/v1/salesbatch')
def predict_sales_batch():
  # get the uploaded CSV file from the request
  file = request.files['file']

  # read the file into a DataFrame
  input_data = pd.read_csv(file)

  input_data['log_Product_Allocated_Area'] = np.log(input_data['Product_Allocated_Area'])
  input_data = input_data.drop(columns=['Product_Allocated_Area'])
  input_data['Store_Establishment_Year'] = input_data['Store_Establishment_Year'].astype('object')

  # Make predictions for the batch data
  predictions = model.predict(input_data.drop("Product_Id",axis=1)).tolist()

  product_id_list = input_data.Product_Id.values.tolist()
  output_dict = dict(zip(product_id_list, predictions))

  return output_dict

# Run the Flask app in debug mode
if __name__ == '__main__':
  app.run(debug=True)


### Dependencies Files

In [ ]:
%%writefile requirements.txt
pandas==2.2.2
numpy==2.0.2
scikit-learn==1.6.1
xgboost==2.1.4
joblib==1.4.2
Werkzeug==2.2.2
flask==2.2.2
gunicorn==20.1.0
requests==2.32.4
uvicorn[standard]

### Dockerfile

In [ ]:
%%writefile Dockerfile
FROM python:3.9-slim

# Set the working directory inside the container
WORKDIR /app

# Copy all files from the current directory to the container's working directory
COPY . .

# Install dependencies from the requirements file without using cache to reduce image size
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Define the command to start the application using Gunicorn with 4 worker processes
# - `-w 4`: Uses 4 worker processes for handling requests
# - `-b 0.0.0.0:7860`: Binds the server to port 7860 on all network interfaces
# - `app:app`: Runs the Flask app (assuming `app.py` contains the Flask instance named `app`)
CMD ["gunicorn", "-w", "4", "-b", "0.0.0.0:7860", "app:app"]

### Uploading Files to Hugging Face Space for the Backend

In [ ]:
# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

from google.colab import userdata
access_key = userdata.get('hf_token')

repo_id = "MissSamyuktha/sales-forecast-backend"  # Hugging Face space id

# Login to Hugging Face platform with the access token
login(token=access_key)

# initialize the apiA
api = HfApi()

# Upload Flask app files stored in the folder called backend_files
api.upload_folder(folder_path='/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project/backend_files', # Local folder path
                  repo_id=repo_id,    ## Hugging face space id
                  repo_type="space"  # Hugging face repo type "space"
)



In [ ]:
!python app.py

In [ ]:
import json  # To handle JSON formatting for API requests and responses
import requests  # To send HTTP requests to the deployed Flask API

import pandas as pd  # For data manipulation and analysis
import numpy as np  # For numerical computations

In [ ]:
model_root_url = "https://MissSamyuktha-sales-forecast-backend.hf.space"

In [ ]:
model_url = model_root_url + "/v1/sales"  # Endpoint for online (single) inference

In [ ]:
model_batch_url = model_root_url + "/v1/salesbatch"  # Endpoint for batch inference

In [ ]:
payload = {
    "Product_Weight": 120,
    "Product_Allocated_Area": 10,
    "Product_MRP": 200,
    "Product_Sugar_Content": "Regular",
    "Product_Type": "Dairy",
    "Store_Id": "OUT001",
    "Store_Establishment_Year": str(1998),
    "Store_Size": "Medium",
    "Store_Location_City_Type": "Tier 2",
    "Store_Type": "Supermarket Type 1"
}

In [ ]:
# Sending a POST request to the model endpoint with the test payload
response = requests.post(model_url, json=payload)

In [ ]:
response

## Frontend App Deployment

### Setting up a Hugging Face Docker Streamlit Space for the Frontend at [Hugging Face Spaces](https://huggingface.co/spaces/)

# Streamlit UI for App Frontend

In [ ]:
os.chdir("/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project")

In [ ]:
# Create a folder for storing the files needed for frontend UI deployment
os.makedirs("frontend_files", exist_ok=True)

In [ ]:
os.chdir("/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project/frontend_files")

In [ ]:
!pip install streamlit==1.43.2

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import requests

# Set the title of the Streamlit app
st.title("SuperKart Sales Forecasting Application")

# Section for online prediction
st.subheader("Online Sales Prediction")

# collect sales info for sales forecast
Product_Weight = st.number_input("Product Weight", min_value=0, value=100)
Product_Allocated_Area = st.number_input("Product Allocated Area", min_value=0, value=10)
Product_MRP = st.number_input("Product MRP", min_value=0, value=10)
Product_Sugar_Content = st.selectbox("Product Sugar Content", ["Regular", "Low Sugar", "No Sugar"])
Product_Type = st.selectbox("Product Type", ["Frozen Foods", "Dairy", "Canned", "Baking Goods", "Health and Hygiene", "Snack Foods", "Meat", "Household", "Hard Drinks", "Fruits and Vegetables", "Breads", "Soft Drinks", "Breakfast", "Others", "Starchy Foods", "Seafood"])
Store_Id = st.selectbox("Store ID", ["OUT001", "OUT002", "OUT003", "OUT004"])
Store_Establishment_Year = st.selectbox("Store Establishment Year", ["1987", "1998", "1999", "2009"])
Store_Size = st.selectbox("Store Size", ["Small", "Medium", "High"])
Store_Location_City_Type = st.selectbox("City Type(Tier)", ["Tier 1", "Tier 2", "Tier 3"])
Store_Type = st.selectbox("Store Type", ["Supermarket Type 1", "Supermarket Type 2", "Departmental Store", "Food Mart"])

# Convert user input into a DataFrame
input_data = pd.DataFrame([{
    'Product_Weight': Product_Weight,
    'log_Product_Allocated_Area': np.log(Product_Allocated_Area),
    'Product_MRP': Product_MRP,
    'Product_Sugar_Content': Product_Sugar_Content,
    'Product_Type': Product_Type,
    'Store_Id': Store_Id,
    'Store_Establishment_Year': str(Store_Establishment_Year),
    'Store_Size': Store_Size,
    'Store_Location_City_Type': Store_Location_City_Type,
    'Store_Type': Store_Type
}])

# Make prediction when the "Predict" button is clicked
if st.button("Predict"):
    response = requests.post("https://MissSamyuktha-sales-forecast-backend.hf.space/v1/sales", json=input_data.to_dict(orient='records')[0])  # Send data to Flask API
    if response.status_code == 200:
        result = response.json()
        prediction = result["Forecasted Sales"]
        st.success(f"Predicted Sales (in dollars): {prediction}")
    else:
        st.error("Error making prediction.")

# Section for batch prediction
st.subheader("Batch Prediction")

# Allow users to upload a CSV file for batch prediction
uploaded_file = st.file_uploader("Upload CSV file for batch prediction", type=["csv"])

# Make batch prediction when the "Predict Batch" button is clicked
if uploaded_file is not None:
    if st.button("Predict Batch"):
        response = requests.post("https://MissSamyuktha-sales-forecast-backend.hf.space/v1/salesbatch", files={"file": uploaded_file})  # Send file to Flask API
        if response.status_code == 200:
            predictions = response.json()
            st.success("Batch predictions completed!")
            st.write(predictions)  # Display the predictions
        else:
            st.error("Error making batch prediction.")

### Dependencies File

In [ ]:
%%writefile requirements.txt
pandas==2.2.2
numpy==2.0.2
requests==2.32.4
streamlit==1.43.2

### Dockerfile

In [ ]:
%%writefile Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9-slim

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

# Run the Streamlit app on port 7860 (required by Hugging Face Spaces)
CMD ["streamlit", "run", "app.py", "--server.port=7860", "--server.address=0.0.0.0"]

### Uploading Files to Hugging Face Space for the Frontend

In [ ]:
# for hugging face space authentication to upload files
from huggingface_hub import HfApi

repo_id = "MissSamyuktha/sales-forecast-frontend"  #  Hugging Face space id

# Initialize the API
api = HfApi()

# Upload Streamlit app files stored in the folder called deployment_files
api.upload_folder(
    folder_path="/content/drive/MyDrive/PGP - AI ML/Model Deployment/Sales Forecasting Project/frontend_files",  # Local folder path
    repo_id=repo_id,  # Hugging face space id
    repo_type="space",  # Hugging face repo type "space"
)

In [ ]:
!streamlit run app.py --server.port=7860

In [ ]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(7860, {'cache': false})"))


# Deployed Application Links on HuggingFace

Backend Flask App: [Backend Flask App](https://huggingface.co/spaces/MissSamyuktha/sales-forecast-backend)

Streamlit App: [Frontend Streamlit UI](https://huggingface.co/spaces/MissSamyuktha/sales-forecast-frontend)


# Key Insights and Recommendations

## Actionable Insights

- **Key Sales Drivers**:
  - `Product_MRP` and `Product_Weight` are good predictors of revenue.
  - `Store_Type` and `Store_Size` influence performance.
  - Tier 1 cities lead in median sales, while Tier 2 cities contribute heavily through volume.
  
- **Model Performance**:
  - Tuned **XGBoost** model achieved:
    - R² ≈ 92.9% on test data
    - RMSE ≈ 284.75
  - Indicates strong predictive accuracy and generalization.

- **Feature Importance**:
  - Top contributors across models include:
    - `Product_MRP`
    - `Product_Weight`
    - `Store_Size_Small`
    - `Store_Id_OUT002`
    - `Store_Type_Food Mart`

## Recommendations

### Modeling Strategy
- Retrain quarterly to capture seasonality and promotional effects.
- Apply log transformation to skewed features like `Product_Allocated_Area`.

### 🔹 Deployment & Integration
- Backend API deployed via Flask on Hugging Face Spaces, ensures uptime and monitor logs.
- Streamlit frontend enables interactive single and batch predictions.
- Batch endpoint supports regional planning and bulk forecasting.

### Business Impact
- Optimize inventory and reduce overstocking.
- Align marketing with high-performing products and store types.
- Benchmark store performance using forecasted vs actual sales.

##  Next Steps
- Integrate API into SuperKart’s internal dashboards or ERP system.
- Automate data pipelines for retraining and prediction.
- Expand to multi-quarter forecasting and include external factors (e.g., holidays, weather).
